In [ ]:
# ═══════════════════════════════════════════════
# STABLE MONEY AI AVATAR — GPU + Wav2Lip
# ═══════════════════════════════════════════════
import subprocess, os, sys, time, threading

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG — Set your keys here
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GROQ_API_KEY = "YOUR_GROQ_API_KEY"       # ← Paste your Groq API key
NGROK_TOKEN  = "YOUR_NGROK_TOKEN"        # ← Paste your ngrok authtoken

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# ── STEP 1: Install system deps ──
print("📦 Installing system dependencies...")
os.system("apt-get update -qq && apt-get install -y -qq ffmpeg espeak-ng zstd > /dev/null 2>&1")
print("✅ System deps installed")

# ── STEP 2: Install Python deps ──
print("📦 Installing Python packages...")
os.system("pip install -q fastapi 'uvicorn[standard]' python-multipart aiofiles")
os.system("pip install -q edge-tts groq pyngrok nest_asyncio")
os.system("pip install -q opencv-python-headless librosa")
print("✅ Python packages installed")

# ── STEP 3: GPU check ──
import torch
HAS_GPU = torch.cuda.is_available()
print(f"🖥️  GPU: {torch.cuda.get_device_name(0) if HAS_GPU else 'CPU only'}")

# ── STEP 4: Clone Wav2Lip + download checkpoint ──
if HAS_GPU:
    print("📦 Setting up Wav2Lip...")
    if not os.path.exists("/content/Wav2Lip"):
        os.system("git clone https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip")
    os.makedirs("/content/Wav2Lip/checkpoints", exist_ok=True)
    CKPT = "/content/Wav2Lip/checkpoints/wav2lip_gan.pth"
    if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 1000000:
        print("⬇️  Downloading Wav2Lip GAN checkpoint...")
        os.system("pip install -q gdown")
        import gdown
        # Official Wav2Lip GAN checkpoint from authors
        gdown.download("https://drive.google.com/uc?id=1qx25dO0MXsWECgSi4VXmFJxkdJgyiksP", CKPT, quiet=False)
        if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 1000000:
            # Fallback: try alternate ID
            gdown.download("https://drive.google.com/uc?id=1PpNKqq6WS0TieGU9i2t7k2e3bTjHlylz", CKPT, quiet=False)
    if os.path.exists(CKPT):
        print(f"✅ Wav2Lip checkpoint: {os.path.getsize(CKPT) / 1e6:.1f} MB")
    else:
        print("⚠️  Wav2Lip checkpoint download failed — will run audio-only")

    # Install Wav2Lip deps
    os.system("pip install -q face_alignment")

    # Symlink checkpoint to working directory
    os.makedirs("/content/checkpoints", exist_ok=True)
    if not os.path.exists("/content/checkpoints/wav2lip_gan.pth"):
        os.symlink(CKPT, "/content/checkpoints/wav2lip_gan.pth")
else:
    print("⚠️  No GPU — Wav2Lip disabled, audio-only mode")

# ── STEP 5: Download latest code from repo ──
print("📥 Downloading latest code from GitHub...")
REPO = "https://raw.githubusercontent.com/aayushjha-blip/stable-money-avatar/main"

os.makedirs("/content/static", exist_ok=True)

os.system(f'curl -sL {REPO}/server.py -o /content/server.py')
os.system(f'curl -sL {REPO}/static/index.html -o /content/static/index.html')
os.system(f'curl -sL {REPO}/knowledge_base.txt -o /content/knowledge_base.txt')

# Download avatar image
if not os.path.exists("/content/static/mugshot.jpg"):
    os.system(f'curl -sL {REPO}/static/mugshot.jpg -o /content/static/mugshot.jpg')
# Wav2Lip needs avatar.jpg in root
if not os.path.exists("/content/avatar.jpg") and os.path.exists("/content/static/mugshot.jpg"):
    os.system("cp /content/static/mugshot.jpg /content/avatar.jpg")

print(f"✅ server.py: {os.path.getsize('/content/server.py')} bytes")
print(f"✅ index.html: {os.path.getsize('/content/static/index.html')} bytes")
print(f"✅ knowledge_base.txt: {os.path.getsize('/content/knowledge_base.txt')} bytes")

# ── STEP 6: Configure ngrok ──
from pyngrok import conf, ngrok
conf.get_default().auth_token = NGROK_TOKEN
print("✅ ngrok configured")

# ── STEP 7: Kill any existing server ──
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
time.sleep(2)

# ── STEP 8: Start ngrok + server ──
import nest_asyncio, uvicorn
nest_asyncio.apply()

os.chdir("/content")
sys.path.insert(0, "/content")
if HAS_GPU:
    sys.path.insert(0, "/content/Wav2Lip")

ngrok.kill()
time.sleep(1)
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("=" * 55)
print("🌐  PUBLIC URL:", public_url)
print("🔌  WS URL:", public_url.replace("https://", "wss://") + "/ws/talk")
print("=" * 55)

from server import app as fastapi_app

config = uvicorn.Config(fastapi_app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
server.install_signal_handlers = lambda: None

thread = threading.Thread(target=server.run)
thread.daemon = True
thread.start()

print("✅ Server running! Open the PUBLIC URL above in your browser.")
time.sleep(3)

# Keep cell alive
while True:
    time.sleep(60)
